# Fine-Tuning DistilBERT for Emotion Classification

In this notebook I fine-tune **DistilBERT**, a pretrained Transformer model, so that it can read a short English sentence and tell which emotion it expresses: **sadness, joy, love, anger, fear or surprise**.

For example:

| Sentence | Emotion |
|---|---|
| "i am feeling grouchy" | anger |
| "i left with my bouquet of tulips feeling slightly more optimistic" | joy |

This kind of task is called **text classification**, and when the categories are feelings it is called **sentiment analysis**. Companies use it to sort customer reviews, route angry support tickets, or follow reactions to a product on social media.

**Why fine-tune instead of training from scratch?** Teaching a model English from zero needs billions of sentences and a lot of computing power. DistilBERT has already read a huge amount of English text and understands how words relate to each other. It just doesn't know my six emotions yet. Fine-tuning means training it a little more on my specific task, like hiring someone who already speaks fluent English and training them for a week on a new job.

The notebook follows the full workflow:

```
Load data -> Load model -> Tokenize -> Pad batches -> Metrics -> Training setup -> Train -> Inference
   (0)          (1)          (2)          (3)           (4)           (5)          (6)       (7)
```

Each task has three parts: an explanation, a `%%writefile` cell that saves the task file exactly as it is submitted, and a demo cell that runs the project's main file.

**Before running:** go to **Runtime -> Change runtime type -> T4 GPU**. Training on a CPU would take hours.

## Setup

### Install the required versions

I install the package versions the project requires. Colab already has a GPU version of PyTorch, so I keep it instead of the CPU-only version listed in the requirements (that one is for the checker's machine).

In [ ]:
!pip install -q transformers==4.56.1 datasets==4.0.0 accelerate==1.12.0 \
    sentencepiece==0.2.1 tabulate==0.9.0

In [ ]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

### Log in to Hugging Face

At the end of training, the model is pushed to the Hugging Face Hub, so I need to be logged in. I store my access token in Colab's **Secrets** panel (the key icon on the left) under the name `HF_TOKEN`, with notebook access turned on. This way the token never appears in the notebook, which matters because the repository is public. The token needs **Write** permission.

I also turn off Weights & Biases logging. If the `wandb` package is installed, the Trainer would otherwise stop and ask for a W&B account.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

# Stop the Trainer from asking for a Weights & Biases account
os.environ["WANDB_MODE"] = "disabled"

# Part 1: The Data

## Task 0: Load the Dataset

**What I'm doing:** loading the examples the model will learn from.

The [Emotion dataset](https://huggingface.co/datasets/dair-ai/emotion) has 20,000 short English sentences, each labeled with one of six emotions. The labels are stored as numbers from 0 to 5.

It comes in three splits, and each one has a different job:

| Split | Size | Job |
|---|---|---|
| train | 16,000 | the model learns from these |
| validation | 2,000 | checks progress during training on sentences the model hasn't learned from |
| test | 2,000 | used once at the end as the final, honest score |

It's like studying from a textbook, taking practice exams along the way, and then taking the real exam on questions you've never seen.

`datasets.load_dataset()` downloads the data from the Hugging Face Hub and caches it, so later runs load instantly. It returns a `DatasetDict`, which works like a Python dictionary: `{"train": ..., "validation": ..., "test": ...}`. The `"split"` argument picks the version of the dataset that is already divided into these three parts.

The checker only allows plain `import datasets` (no `from` imports), so I call the function as `datasets.load_dataset`.

In [ ]:
%%writefile 0-load_dataset.py
#!/usr/bin/env python3
"""Load the Emotion dataset for sentiment analysis."""
import datasets


def load_emotion_dataset():
    """
    Load the dair-ai/emotion dataset from the Hugging Face Hub.

    Returns:
        DatasetDict: the train, validation and test splits
    """
    return datasets.load_dataset("dair-ai/emotion", "split")

### Dataset structure

In [ ]:
load_emotion_dataset = __import__('0-load_dataset').load_emotion_dataset

dataset = load_emotion_dataset()

print(type(dataset))

print("\nDataset Structure:")
print(dataset)

print("\nDataset Column Names:")
print(dataset.column_names)

print("\nDataset Shape:")
print(dataset.shape)

### Label distribution and examples

Here I count how many sentences belong to each emotion in each split. The classes are **imbalanced**: joy is about a third of the training data, while surprise is under 4%. This is important later, because it's the reason I use **weighted** metrics instead of plain accuracy.

In [ ]:
import numpy as np

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

for split_name in dataset.keys():
    print(f"\n{split_name.capitalize()} Set Label Distribution:")
    split_labels = np.array(dataset[split_name]['label'])
    for idx, name in enumerate(label_names):
        count = np.sum(split_labels == idx)
        percentage = 100 * count / len(split_labels)
        print(f"  {name:<10}: {count} samples ({percentage:.2f}%)")

    print(f"\nSample Examples from {split_name.capitalize()} Set:")
    for i in range(min(5, len(dataset[split_name]))):
        text = dataset[split_name]['text'][i]
        label_id = dataset[split_name]['label'][i]
        emotion = label_names[label_id]
        print(f"{i+1}. [{emotion.upper()}] {text}")

The label names are also stored inside the dataset itself, because the `label` column is a `ClassLabel` feature:

In [ ]:
print(dataset["train"].features["label"].names)

# Part 2: The Model and the Preprocessing

## Task 1: Load the Pretrained Tokenizer and Model

**What I'm doing:** loading the pretrained "brain" and the tool that translates text into numbers it understands.

**The tokenizer.** Models can't read letters, only numbers. The tokenizer turns text into **token IDs** using a fixed vocabulary of 30,522 pieces. DistilBERT uses **subword tokenization** (WordPiece): common words stay whole, and rare words are split into smaller pieces it knows, for example `"didnt"` becomes `didn` + `##t`. This way the tokenizer never meets a completely unknown word.

The tokenizer produces two inputs for the model:
- `input_ids`: the token numbers.
- `attention_mask`: 1 for real tokens, 0 for padding, so the model ignores padding.

DistilBERT can read at most **512 tokens**, which is why truncation comes up in the next task.

**The model.** `AutoModelForSequenceClassification` takes the pretrained DistilBERT body and adds a new **classification head** on top. The head turns DistilBERT's understanding of a sentence into six scores, one per emotion. The body already knows English, but the head is brand new with random weights. That's what the warning *"Some weights ... are newly initialized"* means. It is expected, and fine-tuning is the step that trains the head.

The arguments configure the head:
- `num_labels=6` gives six output scores.
- `id2label` and `label2id` are stored in the model's config, so later the model can answer `"joy"` instead of `"1"`.

In [ ]:
%%writefile 1-load_distilbert.py
#!/usr/bin/env python3
"""Load a pretrained DistilBERT tokenizer and classification model."""
import transformers


def load_distilbert(model_name, num_classes, id2label, label2id):
    """
    Load a tokenizer and a sequence classification model.

    Args:
        model_name (str): name of the pretrained DistilBERT model
        num_classes (int): number of output classes
        id2label (dict): maps label ids to label names
        label2id (dict): maps label names to label ids

    Returns:
        tuple: (tokenizer, model)
    """
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    model = transformers.AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        id2label=id2label,
        label2id=label2id
    )
    return tokenizer, model

### Tokenizer details

In [ ]:
load_distilbert = __import__('1-load_distilbert').load_distilbert

model_name = "distilbert-base-uncased"
num_classes = 6
id_to_label = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
label_to_id = {v: k for k, v in id_to_label.items()}

tokenizer, model = load_distilbert(model_name, num_classes, id_to_label, label_to_id)

print("Tokenizer type:", type(tokenizer))
print("\nTokenizer Vocabulary Size:", tokenizer.vocab_size)
print("Tokenizer Max Input Length:", tokenizer.model_max_length)
print("Tokenizer Model Input Names:", tokenizer.model_input_names)

To see subword tokenization in action, I split a few sentences into tokens. Words the tokenizer doesn't know as a whole get split into pieces starting with `##`:

In [ ]:
for sentence in ["i didnt feel humiliated", "I am feeling grouchy"]:
    print(sentence)
    print("  tokens:", tokenizer.tokenize(sentence))
    print("  ids:   ", tokenizer(sentence)["input_ids"])

### Model details

The parameter count can be checked by hand:

| Part | Calculation | Parameters |
|---|---|---|
| Pretrained DistilBERT body | from the checkpoint | 66,362,880 |
| `pre_classifier` (768 -> 768) | 768 x 768 + 768 | 590,592 |
| `classifier` (768 -> 6) | 768 x 6 + 6 | 4,614 |
| **Total** | | **66,958,086** |

All parameters are trainable because nothing is frozen, so the whole network gets fine-tuned.

In [ ]:
print("Model type:", type(model))
print("\nModel Configuration:")
print(model.config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Task 2: Tokenize the Whole Dataset

**What I'm doing:** converting all 20,000 sentences into numbers once, ahead of training.

- `dataset.map()` applies my `tokenize` function to every example in all three splits. It keeps the original columns (`text`, `label`) and adds the new ones (`input_ids`, `attention_mask`).
- `batched=True` sends 1,000 sentences at a time to the tokenizer instead of one by one. The fast tokenizer is written in Rust and processes a whole list in one call, which is much faster.
- `truncation=True` with `max_length` cuts off sentences that are too long.

The tokenizer also adds two special tokens to every sentence: `[CLS]` (id 101) at the start and `[SEP]` (id 102) at the end. Both count toward `max_length`. The model uses `[CLS]` as its summary of the whole sentence, and that summary is what the classification head looks at.

I don't pad here on purpose. Padding happens later, batch by batch, in Task 3.

No imports are needed in this file, because the dataset and the tokenizer come in as arguments.

In [ ]:
%%writefile 2-tokenize_and_map.py
#!/usr/bin/env python3
"""Tokenize the Emotion dataset with a pretrained tokenizer."""


def tokenize_and_map(dataset, tokenizer, max_length, truncation, batched):
    """
    Tokenize the text of every split in the dataset.

    Args:
        dataset (DatasetDict): train, validation and test splits
        tokenizer: pretrained tokenizer
        max_length (int): maximum token length for truncation
        truncation (bool): whether to truncate long sequences
        batched (bool): whether to process examples in batches

    Returns:
        tuple: tokenized train, validation and test splits
    """
    def tokenize(examples):
        """Tokenize a batch (or single example) of text."""
        return tokenizer(
            examples["text"],
            truncation=truncation,
            max_length=max_length
        )

    tokenized = dataset.map(tokenize, batched=batched)
    return tokenized["train"], tokenized["validation"], tokenized["test"]

### Demo with `max_length=13`

With a very short limit, truncation becomes visible. The validation sentence is cut after 13 tokens, and the tokenizer still places `[SEP]` (102) at the end. The train sentence only has 7 tokens and is not padded.

In [ ]:
from transformers import logging
logging.set_verbosity_error()

tokenize_and_map = __import__('2-tokenize_and_map').tokenize_and_map

short_train, short_val, short_test = tokenize_and_map(
    dataset,
    tokenizer,
    max_length=13,
    truncation=True,
    batched=True
)

print("First tokenized train example:", short_train[0])
print("First tokenized validation example:", short_val[0])
print("First tokenized test example:", short_test[0])

For training I tokenize again with `max_length=128`, which is long enough for almost every sentence in this dataset.

In [ ]:
tokenized_train, tokenized_val, tokenized_test = tokenize_and_map(
    dataset,
    tokenizer,
    max_length=128,
    truncation=True,
    batched=True
)

print(tokenized_train)

## Task 3: Dynamic Padding

**What I'm doing:** making the sentences in a batch the same length, but only as long as needed.

The model processes sentences in groups (batches), and a batch must be a rectangle of numbers, so shorter sentences are filled with `[PAD]` tokens (id 0). The attention mask marks those positions with 0 so the model ignores them.

There are two ways to pad:
- **Static padding:** pad every sentence to a fixed length like 128. Most emotion sentences are short, so most of each row would be wasted padding.
- **Dynamic padding:** pad each batch only to the length of its own longest sentence.

`DataCollatorWithPadding` does dynamic padding. A **data collator** is the helper that builds each batch: it takes a list of examples, pads them, and turns them into PyTorch tensors. It needs the tokenizer to know which id means `[PAD]`.

In [ ]:
%%writefile 3-dynamic_padding.py
#!/usr/bin/env python3
"""Create a data collator for dynamic padding."""
import transformers


def create_data_collator(tokenizer):
    """
    Set up dynamic padding for tokenized inputs.

    Each batch is padded only to the length of its longest
    sequence, which avoids unnecessary padding.

    Args:
        tokenizer: pretrained tokenizer

    Returns:
        DataCollatorWithPadding: pads inputs dynamically per batch
    """
    return transformers.DataCollatorWithPadding(tokenizer=tokenizer)

### Demo: two batches of four sentences

The first batch is padded to 23 tokens and the second to 30, because those are the lengths of their longest sentences. With static padding, both would be 128 tokens wide.

In [ ]:
create_data_collator = __import__('3-dynamic_padding').create_data_collator

data_collator = create_data_collator(tokenizer)


def get_batch(start_idx, batch_size=4):
    return [
        {k: tokenized_train[i][k] for k in ["input_ids", "attention_mask"]}
        for i in range(start_idx, start_idx + batch_size)
    ]


batch1 = data_collator(get_batch(0))
batch2 = data_collator(get_batch(4))

for i, batch in enumerate([batch1, batch2], 1):
    print(f"Batch {i} shapes:")
    print("  input_ids:", batch['input_ids'].shape)
    print("  Input IDs:")
    print(batch['input_ids'])
    print("  attention_mask:", batch['attention_mask'].shape)
    print()

    print(f"  Decoded texts from Batch {i}:")
    for j, ids in enumerate(batch['input_ids']):
        decoded_text = tokenizer.decode(ids, skip_special_tokens=True)
        print(f"    Example {j+1}: {decoded_text}")
    print()

    print("  Decoded texts (with special tokens):")
    for j, ids in enumerate(batch['input_ids']):
        decoded_text = tokenizer.decode(ids, skip_special_tokens=False)
        print(f"    Example {j+1}: {decoded_text}")
    print()

# Part 3: Training

## Task 4: Evaluation Metrics

**What I'm doing:** writing the function that measures how good the model is. The Trainer calls it after every evaluation.

**Logits.** For each sentence, the model outputs one raw score per class, called **logits**. The highest score is the prediction, so I use `np.argmax(logits, axis=-1)` to pick the position of the highest score in each row.

The Trainer passes an `EvalPrediction` object with two attributes: `predictions` (the logits) and `label_ids` (the true answers).

**The metrics:**
- **Accuracy:** correct predictions out of all predictions.
- **Precision** (per class): when the model predicted this class, how often was it right?
- **Recall** (per class): of all sentences that truly belong to this class, how many did the model find?
- **F1:** balances precision and recall. It's only high when both are high.

**Weighted average.** Each class's score is weighted by its **support**, the number of true examples in that class. Because the emotion classes are imbalanced, this gives a fair overall score, and precision and recall show *how* the model is wrong in a way accuracy alone can't.

Two small details in the code:
- `zero_division=0` avoids a warning if the model never predicts a class (common early in training for a rare class like surprise).
- `float(...)` converts NumPy numbers to plain Python floats, so the dictionary prints cleanly.

In [ ]:
%%writefile 4-compute_metrics.py
#!/usr/bin/env python3
"""Compute evaluation metrics for a classification model."""
import numpy as np
import sklearn.metrics


def compute_metrics(predictions):
    """
    Compute accuracy, precision, recall and F1-score.

    Args:
        predictions: EvalPrediction object with model logits in
            `predictions` and true labels in `label_ids`

    Returns:
        dict: accuracy, weighted precision, weighted recall
            and weighted F1-score
    """
    logits = predictions.predictions
    labels = predictions.label_ids
    preds = np.argmax(logits, axis=-1)

    accuracy = sklearn.metrics.accuracy_score(labels, preds)
    precision = sklearn.metrics.precision_score(
        labels, preds, average="weighted", zero_division=0
    )
    recall = sklearn.metrics.recall_score(
        labels, preds, average="weighted", zero_division=0
    )
    f1 = sklearn.metrics.f1_score(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }

### Demo with a small example

Five sentences and three classes. The predictions are `[0, 1, 2, 1, 2]` and the true labels are `[0, 1, 2, 0, 1]`.

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| 0 | 1/1 = 1.0 | 1/2 = 0.5 | 0.667 | 2 |
| 1 | 1/2 = 0.5 | 1/2 = 0.5 | 0.5 | 2 |
| 2 | 1/2 = 0.5 | 1/1 = 1.0 | 0.667 | 1 |

Weighted precision = (2 x 1.0 + 2 x 0.5 + 1 x 0.5) / 5 = **0.7**, weighted recall = **0.6**, weighted F1 = **0.6**, and accuracy = 3/5 = **0.6**.

In [ ]:
compute_metrics = __import__('4-compute_metrics').compute_metrics
from types import SimpleNamespace

logits = np.array([
    [2.0, 1.0, 0.1],
    [0.1, 1.5, 0.4],
    [2.0, 0.2, 3.0],
    [0.5, 1.0, 0.3],
    [0.2, 0.1, 2.0]
])
labels = np.array([0, 1, 2, 0, 1])

pred = SimpleNamespace(predictions=logits, label_ids=labels)
print("Predicted classes:", np.argmax(logits, axis=-1))
print(compute_metrics(pred))

## Task 5: Configure the Training Arguments

**What I'm doing:** writing down the training settings. `TrainingArguments` is like a settings sheet that the Trainer follows.

**The hyperparameters passed in:**

| Setting | Value | Meaning |
|---|---|---|
| `num_train_epochs` | 5 | the model sees all 16,000 training sentences 5 times |
| `per_device_train_batch_size` | 16 | it learns from 16 sentences at a time (1,000 steps per epoch) |
| `per_device_eval_batch_size` | 32 | evaluation needs less memory per sentence, so batches can be bigger |
| `learning_rate` | 2e-5 | very small steps, so fine-tuning doesn't destroy what DistilBERT already knows |
| `weight_decay` | 0.01 | regularization: gently pushes weights toward zero to reduce overfitting |
| `seed` | 0 | fixes the randomness so results can be reproduced |

**The settings the function adds itself:**
- `eval_strategy="epoch"` and `save_strategy="epoch"`: evaluate on the validation set and save a checkpoint at the end of every epoch.
- `load_best_model_at_end=True` with `metric_for_best_model="f1"`: after training, keep the checkpoint with the highest validation F1, not just the last one.
- `greater_is_better=True`: a higher F1 is better. (For loss it would be `False`.)
- `push_to_hub=True`: upload the model to the Hugging Face Hub.

Older tutorials use `evaluation_strategy`, but it was renamed to `eval_strategy` in recent versions of Transformers.

In [ ]:
%%writefile 5-configure_training_arguments.py
#!/usr/bin/env python3
"""Configure training arguments for fine-tuning a model."""
import transformers


def configure_training_args(output_dir, epochs, per_device_train_batch_size,
                            per_device_eval_batch_size, learning_rate,
                            weight_decay, metric_for_best_model, seed):
    """
    Set up the training arguments for a Hugging Face Trainer.

    Evaluation and checkpoint saving happen at the end of each
    epoch, the best checkpoint is loaded at the end of training,
    and the model is set up to be pushed to the Hugging Face Hub.

    Args:
        output_dir (str): directory for checkpoints and logs
        epochs (int): number of training epochs
        per_device_train_batch_size (int): training batch size
        per_device_eval_batch_size (int): evaluation batch size
        learning_rate (float): learning rate for the optimizer
        weight_decay (float): weight decay for regularization
        metric_for_best_model (str): metric to pick the best checkpoint
        seed (int): random seed for reproducibility

    Returns:
        transformers.TrainingArguments: the training configuration
    """
    return transformers.TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=True,
        push_to_hub=True,
        seed=seed
    )

In [ ]:
configure_training_args = __import__('5-configure_training_arguments').configure_training_args

training_args = configure_training_args(
    output_dir="./hbtn_emotion_classifier",
    epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    metric_for_best_model="f1",
    seed=0
)

# The full printout is very long, so I show only the settings I chose
for name in ["output_dir", "num_train_epochs", "per_device_train_batch_size",
             "per_device_eval_batch_size", "learning_rate", "weight_decay",
             "eval_strategy", "save_strategy", "load_best_model_at_end",
             "metric_for_best_model", "greater_is_better", "push_to_hub", "seed"]:
    print(f"{name:<30} {getattr(training_args, name)}")

## Task 6: Train, Evaluate and Save the Model

**What I'm doing:** the actual fine-tuning. The `Trainer` connects all the earlier pieces and runs the training loop for me.

**What happens at every training step:**
1. Take 16 sentences and pad them into a batch with the data collator.
2. Pass them through the model to get logits (the **forward pass**).
3. Compare the logits with the true labels and compute the **loss**, a number that measures how wrong the model was.
4. Work out how each of the 67 million weights contributed to the error (**backpropagation**).
5. Adjust the weights slightly to reduce the error (the AdamW optimizer, using my learning rate and weight decay).

**At the end of every epoch:** evaluate on the validation set with `compute_metrics`, save a checkpoint, and push it to the Hub.

**After the last epoch:** reload the checkpoint with the best validation F1.

**Then my function:**
- evaluates on the **test set**, sentences never used for learning or for picking the best checkpoint, so this is the honest final score;
- saves the model and the tokenizer together, because the model only understands token IDs from its own tokenizer;
- returns the trainer, the training results and the test results.

`processing_class=tokenizer` is the current name for the old `tokenizer=` argument.

In [ ]:
%%writefile 6-training_mode.py
#!/usr/bin/env python3
"""Train, evaluate and save a Hugging Face model."""
import transformers


def train_model(model, training_args, train_dataset, eval_dataset,
                test_dataset, tokenizer, data_collator, compute_metrics,
                model_save_name):
    """
    Fine-tune a model, evaluate it and save it with its tokenizer.

    Args:
        model (transformers.PreTrainedModel): model to fine-tune
        training_args (transformers.TrainingArguments): training setup
        train_dataset (datasets.Dataset): tokenized training data
        eval_dataset (datasets.Dataset): tokenized validation data
        test_dataset (datasets.Dataset): tokenized test data
        tokenizer: tokenizer used for preprocessing
        data_collator (callable): data collator for dynamic padding
        compute_metrics (callable): computes metrics from predictions
        model_save_name (str): directory to save the model and tokenizer

    Returns:
        tuple: (trainer, train_results, test_results)
    """
    trainer = transformers.Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    train_results = trainer.train()
    test_results = trainer.evaluate(eval_dataset=test_dataset)

    trainer.save_model(model_save_name)
    tokenizer.save_pretrained(model_save_name)

    return trainer, train_results, test_results

### Launch the fine-tuning

I reload a fresh model with the seed fixed, so the classification head starts from the same random weights every time. On a T4 GPU this takes roughly 10 to 15 minutes.

In [ ]:
from transformers import set_seed
set_seed(0)

train_model = __import__('6-training_mode').train_model

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
label_to_id = {name: idx for idx, name in enumerate(label_names)}
id_to_label = {idx: name for idx, name in enumerate(label_names)}

tokenizer, model = load_distilbert(model_name, len(label_names), id_to_label, label_to_id)

trainer, train_results, test_results = train_model(
    model=model,
    training_args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    test_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    model_save_name="./hbtn_emotion_classifier_best_model"
)

print("\n Training Results:")
print(f"  Total training time: {train_results.metrics['train_runtime']:.2f} seconds")
print(f"  Training loss: {train_results.training_loss:.4f}")
print(f"  Training samples/sec: {train_results.metrics['train_samples_per_second']:.2f}")
print(f"  Training steps/sec: {train_results.metrics['train_steps_per_second']:.2f}")

print("\n Test Set Performance:")
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1 Score:  {test_results['eval_f1']:.4f}")
print(f"  Loss:      {test_results['eval_loss']:.4f}")

print("\n Model and tokenizer saved to:", "./hbtn_emotion_classifier_best_model")

### Validation results per epoch

The Trainer keeps a log of every evaluation. This table shows how the validation metrics changed from epoch to epoch, and which epoch was kept as the best model.

In [ ]:
import pandas as pd

history = [log for log in trainer.state.log_history if "eval_f1" in log]
history_df = pd.DataFrame(history)[["epoch", "eval_loss", "eval_accuracy", "eval_f1"]]
print(history_df.to_string(index=False))
print("\nBest checkpoint:", trainer.state.best_model_checkpoint)

### Detailed test set evaluation

**Confusion matrix:** each row is the true emotion and each column is the predicted emotion. The diagonal holds the correct predictions, and every number off the diagonal is a mistake, showing exactly which emotions get mixed up.

**Classification report:** precision, recall and F1 for each emotion separately, plus the weighted averages. This is where weaknesses on small classes like love and surprise become visible.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_output = trainer.predict(tokenized_test)
pred_labels = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

conf_matrix = confusion_matrix(true_labels, pred_labels)
print("Confusion Matrix:")
print("\nPredicted ->")
print("Actual |")
conf_df = pd.DataFrame(conf_matrix, index=label_names, columns=label_names)
print(conf_df)

report = classification_report(true_labels, pred_labels, target_names=label_names, digits=4)
print("\nClassification Report:\n", report)

The same confusion matrix as a heatmap, which makes the mix-ups easier to spot:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(conf_matrix, cmap="Blues")
ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=45)
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted emotion")
ax.set_ylabel("True emotion")
ax.set_title("Confusion Matrix on the Test Set")

for i in range(len(label_names)):
    for j in range(len(label_names)):
        color = "white" if conf_matrix[i, j] > conf_matrix.max() / 2 else "black"
        ax.text(j, i, conf_matrix[i, j], ha="center", va="center", color=color)

fig.colorbar(im)
plt.tight_layout()
plt.show()

After training, the model should be visible on the Hugging Face Hub at `huggingface.co/<your-username>/hbtn_emotion_classifier`. The repository name comes from `output_dir` in the training arguments.

# Part 4: Using the Model

## Task 7: Inference Mode

**What I'm doing:** using the trained model on new sentences.

To classify a sentence by hand, I would have to tokenize it, run the model without tracking gradients, pick the highest logit with `argmax`, and look up the label name. A Hugging Face **inference pipeline** packages all these steps into one object: raw text goes in, and emotion names with scores come out.

- `"text-classification"` sets the task: one label per piece of text.
- `model=model_path` and `tokenizer=model_path` load my fine-tuned model and its tokenizer from the folder saved in Task 6. The label names come from the model's config, which is why I get `'sadness'` instead of `'LABEL_0'`.
- `top_k` sets how many predictions to return per sentence. The scores come from **softmax**, so all six add up to 1.

`model_path` could also be my Hub repository name, and the pipeline would download the model from the Hub.

In [ ]:
%%writefile 7-inference_mode.py
#!/usr/bin/env python3
"""Create a text classification pipeline from a fine-tuned model."""
import transformers


def inference_mode(model_path, top_k):
    """
    Initialize a text classification pipeline.

    Args:
        model_path (str): path to the saved model and tokenizer
        top_k (int): number of top predictions to return per input

    Returns:
        transformers.Pipeline: pipeline ready to classify new texts
    """
    return transformers.pipeline(
        "text-classification",
        model=model_path,
        tokenizer=model_path,
        top_k=top_k
    )

### Before fine-tuning

To see what fine-tuning changed, I first classify five sentences with a fresh DistilBERT whose classification head is still random. This is the manual version of what the pipeline does. The predictions are close to random guesses.

In [ ]:
set_seed(0)

texts = [
    "I feel really so disappointed.",
    "I absolutely love spending time with my family.",
    "This is so frustrating! I'm really angry!",
    "I love this new song!",
    "I'm Petrified about what might happen."
]

base_tokenizer, base_model = load_distilbert(model_name, len(label_names), id_to_label, label_to_id)

base_model.eval()
for text in texts:
    inputs = base_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

    with torch.no_grad():
        outputs = base_model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=-1).item()

    print(f"Text: {text}")
    print(f"Prediction: {base_model.config.id2label[predicted_class]}")
    print("-" * 40)

### After fine-tuning: simple vocabulary

The same five sentences with my fine-tuned model, showing the three most likely emotions for each.

In [ ]:
inference_mode = __import__('7-inference_mode').inference_mode

model_path = "./hbtn_emotion_classifier_best_model"

emotion_classifier = inference_mode(model_path, top_k=3)
predictions = emotion_classifier(texts)

for text, pred in zip(texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("-" * 40)

### After fine-tuning: complex vocabulary

Words like "exasperating", "indignation" and "apprehension" are rare in the Emotion training data. If the model still gets these right, it shows that the knowledge from DistilBERT's pretraining carried over: fine-tuning only taught it how to connect what it already knew to my six labels.

In [ ]:
complex_texts = [
    "I am profoundly disenfranchised and disillusioned by the outcome.",
    "I derive immense gratification and elation from familial interactions.",
    "The current state of affairs is entirely exasperating; my indignation is substantial.",
    "This novel musical composition captivates me entirely.",
    "I am utterly paralyzed by an overwhelming apprehension regarding potential eventualities."
]

emotion_classifier = inference_mode(model_path, top_k=1)
predictions = emotion_classifier(complex_texts)

for text, pred in zip(complex_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("-" * 40)

### Try your own sentences

In [ ]:
my_texts = [
    "I can't wait for the weekend!",
    "I miss my old friends so much.",
]

for text, pred in zip(my_texts, inference_mode(model_path, top_k=2)(my_texts)):
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("-" * 40)

# Key Findings

- **Fine-tuning is essential.** Before fine-tuning, the random classification head produced near-random predictions. After fine-tuning, the same sentences were classified correctly with high confidence.
- **Pretrained knowledge transfers.** The model handled rare, complex vocabulary that barely appears in the training data, thanks to DistilBERT's pretraining on general English text.
- **Minority classes are the hardest.** Love and surprise, the two smallest classes, are where most mistakes happen. Weighted metrics and the confusion matrix make this visible where accuracy alone would hide it.
- **Dynamic padding saves computation.** Padding each batch only to its longest sentence (23 or 30 tokens in the demo) instead of a fixed 128 avoids a lot of wasted work.
- **The libraries do the heavy lifting.** Each task file is a short function, but behind it are the dataset loader, the tokenizer, the 67-million-parameter model and the full training loop. The work is in choosing the right tools and settings and understanding what happens inside.

## Author

**Khadija Mustafa** ([@KhadijaTheAnalyst](https://github.com/KhadijaTheAnalyst))